# Supply Chain Optimization
Imports and setup.

In [27]:
import os
import pandas as pd
import numpy as np
import pulp
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

np.random.seed(42)
os.makedirs('outputs', exist_ok=True)

## 1. Data analysis
Load the data and take a first look.

In [28]:
df = pd.read_csv('data/sales_data.csv')
print(df.shape)
df.head()

(76000, 16)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [29]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.dtypes)


print('missing values:', df.isna().sum().sum())
print('duplicates:', df.duplicated().sum())

date                   object
store_id               object
product_id             object
category               object
region                 object
inventory_level         int64
units_sold              int64
units_ordered           int64
price                 float64
discount                int64
weather_condition      object
promotion               int64
competitor_pricing    float64
seasonality            object
epidemic                int64
demand                  int64
dtype: object
missing values: 0
duplicates: 0


Clean the data. Fix dates, fill gaps, remove duplicates and negatives.

In [30]:
df['date']=pd.to_datetime(df['date'], errors='coerce')
df = df.drop_duplicates()
df = df.dropna(subset=['date','store_id','product_id'])

num_cols = ['inventory_level','units_sold','units_ordered','price','discount','competitor_pricing','demand']
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())   # fill gaps with median
    df[c] = df[c].clip(lower = 0)          #no negatives

text_cols = ['category','region','weather_condition','seasonality']
for c in text_cols:
    df[c] = df[c].fillna(df[c].mode()[0]).str.strip().str.title()


print(df.shape)
print(df.isna().sum().sum())

(76000, 16)
0


Standardize units. Everything is in USD, and discount becomes a fraction.

In [31]:
df['discount_pct'] = df['discount']/100
df['net_price']=df['price']*(1-df['discount_pct'])
df['unit_cost'] = df['price']*0.6        # assume cost is 60% of list price
df['month'] = df['date'].dt.to_period('M').astype(str)

df[['price','discount_pct','net_price','unit_cost']].describe()

,price,discount_pct,net_price,unit_cost
count,76000.000000,76000.000000,76000.000000,76000.000000
mean,67.726028,0.090870,61.848853,40.635616
std,39.377899,0.074758,36.977249,23.626739
min,4.740000,0.000000,3.792000,2.844000
25%,31.997500,0.050000,29.021875,19.198500
50%,64.500000,0.100000,58.184000,38.700000
75%,95.830000,0.100000,87.115000,57.498000
max,228.030000,0.250000,228.030000,136.818000


Look at inventory levels and sales vs orders

In [32]:
avg_inv = df.groupby('store_id')['inventory_level'].mean().reset_index()
fig = px.bar(avg_inv, x='store_id', y='inventory_level', title='Average inventory level by store')
fig.show()

monthly = df.groupby('month')[['units_sold','units_ordered']].sum().reset_index()
fig = px.line(monthly, x='month', y=['units_sold','units_ordered'], title='Monthly units sold vs ordered')
fig.show()


# days of stock on hand
df['days_of_stock'] = df['inventory_level']/df['units_sold'].replace(0, np.nan)
print(df['days_of_stock'].describe())
print('overstock rows (>5 days):', (df['days_of_stock']>5).mean().round(3))
print('stockout rows:', (df['inventory_level']==0).mean().round(4))

count    75594.000000
mean         4.301345
std          5.433609
min          1.000000
25%          1.556962
50%          2.853272
75%          5.240908
max        317.500000
Name: days_of_stock, dtype: float64
overstock rows (>5 days): 0.265
stockout rows: 0.0053


The data has no distance or shipping info, so I set up simple assumptions. I use 5 store cities and 3 warehouses (DCs), and compute distances in km.

In [33]:
stores = pd.DataFrame({
    'store_id':['S001','S002','S003','S004','S005'],
    'city':['Islamabad','Karachi','Lahore','Quetta','Peshawar'],
    'lat':[33.68,24.86,31.52,30.18,34.01],
    'lon':[73.05,67.01,74.36,66.99,71.58]})

dcs = pd.DataFrame({
    'dc':['DC_Karachi','DC_Lahore','DC_Islamabad'],
    'lat':[24.86,31.52,33.68],
    'lon':[67.01,74.36,73.05],
    'handling_cost':[0.40,0.55,0.60],      # usd per unit
    'capacity_share':[0.40,0.35,0.35]})    # share of total demand


def haversine_km(lat1,lon1,lat2,lon2):
    r=6371
    p1,p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2-p1
    dl = np.radians(lon2-lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))


dist = pd.DataFrame(index=dcs['dc'], columns=stores['store_id'], dtype=float)
for _,d in dcs.iterrows():
    for _,s in stores.iterrows():
        dist.loc[d['dc'], s['store_id']] = haversine_km(d['lat'],d['lon'],s['lat'],s['lon'])

dist.round(0)

store_id,S001,S002,S003,S004,S005
dc,,,,,
DC_Karachi,1142.0,0.0,1033.0,592.0,1109.0
DC_Lahore,270.0,1033.0,0.0,719.0,380.0
DC_Islamabad,0.0,1142.0,270.0,692.0,141.0


Shipping cost and delivery time per unit. The current setup (baseline) is that each store gets stock from all 3 DCs equally.

In [34]:
BASE_FEE = 1.00        # usd per unit, packing + last mile
RATE = 0.0006          # usd per unit per km
SPEED = 600            # km per day

unit_cost_ship = BASE_FEE + RATE*dist
unit_cost_ship = unit_cost_ship.add(dcs.set_index('dc')['handling_cost'], axis=0)
transit_days = 1 + dist/SPEED

# current setup: every store gets stock from all 3 DCs equally
base_ship = unit_cost_ship.mean(axis=0)
base_days=transit_days.mean(axis=0)

print(base_ship.round(2))
print(base_days.round(2))

store_id
S001    1.80
S002    1.95
S003    1.78
S004    1.92
S005    1.84
dtype: float64
store_id
S001    1.78
S002    2.21
S003    1.72
S004    2.11
S005    1.91
dtype: float64


In [35]:
noise = np.random.normal(0, 0.3, len(df))
df['base_ship_cost'] = df['units_ordered']*df['store_id'].map(base_ship)
df['base_delivery_days'] = (df['store_id'].map(base_days) + noise).clip(lower=0.5)

print(df.groupby('store_id')[['base_ship_cost','base_delivery_days']].agg({'base_ship_cost':'sum','base_delivery_days':'mean'}).round(1))
print('total shipping cost:', round(df['base_ship_cost'].sum()))

          base_ship_cost  base_delivery_days
store_id                                    
S001           2377873.1                 1.8
S002           2715099.3                 2.2
S003           2452329.4                 1.7
S004           2515390.1                 2.1
S005           2516535.4                 1.9
total shipping cost: 12577227
